In [1]:
import pandas as pd
import pyreadstat
import numpy as np
from data_fitting import sample_data_optimized
from data_preprocessing import categorize_age

In [2]:
df_indiv, meta_indiv = pyreadstat.read_dta("../HSE data/r33i_os_84.dta")
df_hh, meta_hh = pyreadstat.read_dta("../HSE data/r33h_os_83.dta")

In [3]:
df_hh.head()

,ccredid_h,ccid_h,bbid_h,aaid_h,zid_h,yid_h,xid_h,wid_h,vid_h,uid_h,...,ccf43_4b,ccf43_4s,ccg1_1,ccg1_2,ccg2,ccg3,ccg4,ccg5,ccg6,ccg7
0,1297.0,10014.0,10014.0,10014.0,10014.0,10014.0,10014.0,10014.0,10014.0,10014.0,...,NaN,,2.0,2.0,1.0,1.0,3.0,3.0,2.0,1.0
1,4853.0,10016.0,10016.0,10016.0,10016.0,10016.0,10016.0,10016.0,10016.0,10016.0,...,NaN,,99999999.0,99999999.0,99999999.0,99999999.0,99999999.0,99999999.0,99999999.0,99999999.0
2,4854.0,10023.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,,2.0,2.0,2.0,1.0,3.0,3.0,2.0,1.0
3,1298.0,10037.0,10037.0,10037.0,10037.0,10037.0,10037.0,10037.0,10037.0,10037.0,...,1.0,Хомяк,2.0,2.0,1.0,1.0,3.0,3.0,2.0,1.0
4,1299.0,10038.0,10038.0,10038.0,10038.0,10038.0,10038.0,10038.0,10038.0,10038.0,...,NaN,,2.0,2.0,1.0,1.0,3.0,3.0,2.0,1.0


## Creating seed

In [4]:
use_cols = {
    "cc_age": "age",
    "cch5": "sex",
    "cc_nfm": "num_members",
    "ccj1": "job",
    "ccj6_2": "work_week",
    "ccj13": "num_colleagues",
    "cck13": "kindergarden",
    "ccj70_2": "study_school",
    "ccj72_5a": "study_university",
    "ccm20_61": "d_heart",
    "ccm20_62": "d_lungs",
    "ccm20_63": "d_liver",
    "ccm20_64": "d_kidneys",
    "ccm20_65": "d_digest",
    "ccm20_66": "d_backbone",
    "ccm20_620": "d_endocrine",
    "ccm20_69": "d_hypertone",
    "ccm20_610": "d_joints",
    "ccm20_611": "d_ent",
    "ccm20_612": "d_neuro",
    "ccm20_613": "d_eyes",
    "ccm20_615": "d_allergy",
    "ccm20_616": "d_veins",
    "ccm20_617": "d_skin",
    "ccm20_618": "d_onco",
    "ccm20_614": "d_gynecol",
    "ccm20_619": "d_genitourin"
    }

In [5]:
seed = df_indiv.merge(df_hh[['ccid_h', 'cc_nfm']], on=['ccid_h'], how='left')[use_cols.keys()]
seed = seed.rename(columns=use_cols).fillna(0)
age_splits = [4, 9, 14, 19, 24, 29, 34, 39, 44, 49, 54, 59, 64, 69, 74, 79, 84]
categorize_age(seed, age_splits, grp_col='age_grp')
seed.loc[:, 'sex'] = pd.to_numeric(seed['sex'], errors='coerce').replace(2, 0).astype('Int64')
seed['fam_size'] = seed['num_members'].astype(int)
seed.loc[seed.fam_size > 5, 'fam_size'] = 5
seed.fam_size -= 1

binary_cols = ['job', 'kindergarden', 'study_school', 'study_university',
               'd_heart', 'd_lungs', 'd_liver', 'd_kidneys', 'd_digest', 'd_backbone',
               'd_endocrine', 'd_hypertone', 'd_joints', 'd_ent', 'd_neuro', 'd_eyes', 'd_allergy',
               'd_veins', 'd_skin', 'd_onco', 'd_gynecol', 'd_genitourin']
for col in binary_cols:
    seed.loc[(seed[col] > 1), col] = 0
    seed[col] = seed[col].astype(int)

In [6]:
seed.head()

,age,sex,num_members,job,work_week,num_colleagues,kindergarden,study_school,study_university,d_heart,...,d_neuro,d_eyes,d_allergy,d_veins,d_skin,d_onco,d_gynecol,d_genitourin,age_grp,fam_size
0,66.0,0.0,3.0,0,0.0,0.0,0,0,0,0,...,0,1,0,0,0,0,0,0,13,2
1,71.5,0.0,1.0,0,0.0,0.0,0,0,0,0,...,0,0,0,0,0,1,1,0,14,0
2,62.5,0.0,1.0,0,0.0,0.0,0,0,0,1,...,0,0,1,0,0,1,1,0,12,0
3,61.0,0.0,3.0,1,99999997.0,150.0,0,0,0,0,...,0,0,0,0,0,0,0,0,12,2
4,55.5,1.0,3.0,1,45.0,99999997.0,0,0,0,0,...,0,0,0,0,0,0,0,0,11,2


## Initializing marginals

In [7]:
def get_marginal_hh(file):
    hh_stats = pd.read_csv(file)
    for i in range(2, 6):
        hh_stats[str(i) + " human"] *= i
    hh_stats["5 human"] += hh_stats["number"]
    return np.array(hh_stats.iloc[0, :5])

# extrapolate marginal statistics up to the given population size
def calibrate(marginal, pop_size):
    marginal_size = marginal.sum()
    extended = (marginal.astype(float) * pop_size / marginal_size).astype(int)
    extended[0] += pop_size - extended.sum()
    return extended

def get_binary_marginals(stats: dict, pop_size):
    columns = []
    marginals = []
    for col, k_true in stats.items():
        columns.append([col])
        marginals.append(np.array([pop_size - k_true, k_true]))
    return columns, marginals

In [8]:
as_stats = pd.read_csv('../HSE data/moscow_stats.csv')
as_stats['total'] = as_stats['total'].astype(int)
as_stats['male'] = as_stats['male'].astype(int)
as_stats['female'] = as_stats['female'].astype(int)
marginal_age_by_sex = np.array([list(as_stats['female']), list(as_stats['male'])], dtype=int)

marginal_hh = get_marginal_hh("../HSE data/moscow_hh_stats.csv")
pop_size = marginal_age_by_sex.sum()
marginal_hh = calibrate(marginal_hh, pop_size)

marginal_job = np.array([
    [0, 3568000], # female
    [0, 3532000]  # male
])
marginal_job[:, 0] = marginal_age_by_sex.sum(axis=1) - marginal_job[:, 1]

other_stats = {
    'kindergarden': 8300,
    'study_school': 1164100,
    'study_university': 815500
}
bin_columns, bin_marginals = get_binary_marginals(other_stats, pop_size)

print(marginal_age_by_sex.sum(), marginal_hh.sum(), marginal_job.sum(), [m.sum() for m in bin_marginals])

13010112 13010112 13010112 [np.int64(13010112), np.int64(13010112), np.int64(13010112)]


In [9]:
seed.fam_size.unique()

array([2, 0, 3, 1, 4])

In [10]:
marginal_hh.shape

(5,)

In [11]:
np.float64(6000000) / np.float64(6000001)

np.float64(0.9999998333333611)

In [9]:
seed.head()

,age,sex,num_members,job,work_week,num_colleagues,kindergarden,study_school,study_university,d_heart,...,d_neuro,d_eyes,d_allergy,d_veins,d_skin,d_onco,d_gynecol,d_genitourin,age_grp,fam_size
0,66.0,0.0,3.0,0,0.0,0.0,0,0,0,0,...,0,1,0,0,0,0,0,0,13,2
1,71.5,0.0,1.0,0,0.0,0.0,0,0,0,0,...,0,0,0,0,0,1,1,0,14,0
2,62.5,0.0,1.0,0,0.0,0.0,0,0,0,1,...,0,0,1,0,0,1,1,0,12,0
3,61.0,0.0,3.0,1,99999997.0,150.0,0,0,0,0,...,0,0,0,0,0,0,0,0,12,2
4,55.5,1.0,3.0,1,45.0,99999997.0,0,0,0,0,...,0,0,0,0,0,0,0,0,11,2


In [10]:
syn_pop = sample_data_optimized(seed,
    [['sex', 'age_grp'], ['fam_size'], ['sex', 'job'], *bin_columns],
    [marginal_age_by_sex, marginal_hh, marginal_job, *bin_marginals])

  0%|          | 3440/13010112 [00:41<43:11:03, 83.66it/s] 


KeyboardInterrupt: 